In [1]:
import pandas as pd
import numpy as np
import warnings
import random

random.seed(42)

# 機器學習相關
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

In [2]:
train_df = pd.read_csv('../data/boy or girl 2025 train_missingValue.csv')
test_df = pd.read_csv('../data/boy or girl 2025 test no ans_missingValue.csv')

train_df['yt'] = pd.to_numeric(train_df['yt'], errors='coerce')
test_df['yt'] = pd.to_numeric(test_df['yt'], errors='coerce')

# 1. 捨棄你之前確認過的無效/文字特徵 (這裡假設 self_intro, star_sign, phone_os 效果不好)
features_to_drop = ['self_intro', 'star_sign', 'phone_os']
train_df = train_df.drop(columns=features_to_drop)
test_df = test_df.drop(columns=features_to_drop)

X = train_df.drop(columns=['id', 'gender'])
y = train_df['gender']
X_test_submission = test_df.drop(columns=['id', 'gender'])
test_ids = test_df['id']

# 2. 【進階特徵工程 1】建立缺失值指示器 (Missing Indicators)
# 在補值之前，先記錄「這個人有沒有填這題」
for col in ['weight', 'height', 'yt']:
    X[f'{col}_is_missing'] = X[col].isnull().astype(int)
    X_test_submission[f'{col}_is_missing'] = X_test_submission[col].isnull().astype(int)

# 3. 過濾極端異常值 (防止 float32 溢位)
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col] = X[col].apply(lambda val: np.nan if pd.notnull(val) and val > 1e15 else val)
        X_test_submission[col] = X_test_submission[col].apply(lambda val: np.nan if pd.notnull(val) and val > 1e15 else val)
        
# 轉換 Target 變數 (讓 XGBoost / LightGBM 可以吃 0, 1)
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

In [3]:
print("開始使用 Random Forest 演算法插補缺失值...")

rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), 
    random_state=42, 
    max_iter=10
)

# 注意：Imputer 產出的是 Numpy Array，我們將它轉回 DataFrame 以方便後續計算 BMI
X_imputed_array = rf_imputer.fit_transform(X)
X_test_imputed_array = rf_imputer.transform(X_test_submission)

# 轉回 Pandas DataFrame
X_imputed = pd.DataFrame(X_imputed_array, columns=X.columns)
X_test_imputed = pd.DataFrame(X_test_imputed_array, columns=X_test_submission.columns)

print("✅ 缺失值插補完成！")

開始使用 Random Forest 演算法插補缺失值...
✅ 缺失值插補完成！


In [4]:
print("開始建立進階衍生特徵 (BMI, Log Transform)...")

def create_advanced_features(df):
    df_new = df.copy()
    
    # 1. 建立 BMI 特徵 (防呆：確保身高不為 0，避免除以零錯誤)
    df_new['height'] = df_new['height'].clip(lower=1) # 身高至少大於 1cm
    df_new['BMI'] = df_new['weight'] / ((df_new['height'] / 100) ** 2)
    
    # 2. 對極端右偏的數值進行 Log1p 轉換 (Log(1+x))
    # clip(lower=0) 確保沒有負數，因為 log 裡面不能是負數
    df_new['yt_log'] = np.log1p(df_new['yt'].clip(lower=0))
    df_new['fb_friends_log'] = np.log1p(df_new['fb_friends'].clip(lower=0))
    
    # 捨棄原本未經 log 轉換的欄位，減少特徵共線性
    df_new = df_new.drop(columns=['yt', 'fb_friends'])
    
    return df_new

X_imputed_eng = create_advanced_features(X_imputed)
X_test_imputed_eng = create_advanced_features(X_test_imputed)

print(f"✅ 特徵工程完成！目前的特徵數量為: {X_imputed_eng.shape[1]}")

開始建立進階衍生特徵 (BMI, Log Transform)...
✅ 特徵工程完成！目前的特徵數量為: 10


In [5]:
print("開始剔除 Permutation Importance 分析出的無效特徵...")

# 將你圖表中重要性 <= 0 的四個特徵名稱填入這裡
useless_features = [
    'BMI', 
    'yt_log', 
    'fb_friends_log', 
    'weight_is_missing' # 或是 yt_is_missing，請依照你的圖表替換
]

# 直接從處理好的資料集中將它們捨棄 (errors='ignore' 確保如果名字打錯也不會報錯中斷)
X_pruned = X_imputed_eng.drop(columns=useless_features, errors='ignore')
X_test_pruned = X_test_imputed_eng.drop(columns=useless_features, errors='ignore')

print(f"✅ 剔除完成！原本有 {X_imputed_eng.shape[1]} 個特徵，現在剩餘 {X_pruned.shape[1]} 個特徵。")

開始剔除 Permutation Importance 分析出的無效特徵...
✅ 剔除完成！原本有 10 個特徵，現在剩餘 6 個特徵。


In [6]:
# 1. 特徵縮放 (改用修剪過特徵的 X_pruned)
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X_pruned)
X_test_scaled = scaler.transform(X_test_pruned)

# 2. 85/15 分層資料切割
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded
)
print("✅ 特徵縮放與資料切割完成！")

✅ 特徵縮放與資料切割完成！


In [7]:
print("初始化三個基礎模型並組裝 Voting Classifier...")

# 1. XGBoost: 強大且泛化能力好
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, max_depth=4, learning_rate=0.05, n_estimators=150)

# 2. LightGBM: 速度快且擅長處理數值區間
lgbm_clf = LGBMClassifier(random_state=42, max_depth=4, learning_rate=0.05, n_estimators=150, verbose=-1)

# 3. Random Forest: 最經典且穩定的樹狀模型
rf_clf = RandomForestClassifier(random_state=42, max_depth=6, n_estimators=150)

# 4. 組合成為軟投票 (Soft Voting) 分類器
# Soft Voting 會將三個模型預測出來的「機率」加總平均，再決定最終類別，通常比單一模型更穩健
ensemble_model = VotingClassifier(
    estimators=[
        ('XGBoost', xgb_clf), 
        ('LightGBM', lgbm_clf), 
        ('RandomForest', rf_clf)
    ], 
    voting='soft'
)

# 5. 進行分層 10-Fold 交叉驗證
print("進行 Stratified 10-Fold 交叉驗證中...")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(ensemble_model, X_train, y_train, cv=skf, scoring='accuracy')

print(f"📊 Ensemble 10-Fold CV 各折準確率: {[round(score, 4) for score in cv_scores]}")
print(f"📊 Ensemble 10-Fold CV 平均準確率: {cv_scores.mean():.4f} (標準差: {cv_scores.std():.4f})\n")

# 6. 使用完整 85% 訓練模型，驗證 15% 驗證集
ensemble_model.fit(X_train, y_train)
y_val_pred = ensemble_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f"🎯 15% 驗證集 (Validation Set) 最終準確率: {val_accuracy:.4f}")

初始化三個基礎模型並組裝 Voting Classifier...
進行 Stratified 10-Fold 交叉驗證中...
📊 Ensemble 10-Fold CV 各折準確率: [np.float64(0.9167), np.float64(0.8889), np.float64(0.8611), np.float64(0.8611), np.float64(0.8611), np.float64(0.9444), np.float64(0.8611), np.float64(0.8056), np.float64(0.8611), np.float64(0.8857)]
📊 Ensemble 10-Fold CV 平均準確率: 0.8747 (標準差: 0.0356)

🎯 15% 驗證集 (Validation Set) 最終準確率: 0.8906


In [8]:
# 1. 預測未知的 Test Set
print("進行測試集預測...")
test_predictions_encoded = ensemble_model.predict(X_test_scaled)

# 2. 轉回原始的 1 (女)、2 (男)
test_predictions_original = le_y.inverse_transform(test_predictions_encoded)

# 3. 組合並存檔
submission_df = pd.DataFrame({
    'id': test_ids,
    'gender': test_predictions_original
})

submission_filename = '../results/submission_advanced_ensemble.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"🏆 預測檔案已成功儲存為：{submission_filename}")

進行測試集預測...
🏆 預測檔案已成功儲存為：submission_advanced_ensemble.csv
